# Notebook to create .npy references used for evaluation of SAR_DDC

For visualization during training and inference, I like to process larger tile of areas cherry picked for their landscape.
This notebook contains the code to create the noisy and denoised references of these tiles.
Currently I have cherry-picked these areas:
- In the "Hamburg" tile (`Hamburg_TDX1_SAR__SSC______SM_S_SRA_20180112T165337_20180112T165345_IMAGE_HH_SRA_strip_004.cos`), the city center arounf the "Alsterfontäne'


In [ ]:
from pathlib import Path
import sys
from omegaconf import OmegaConf
import hydra
import matplotlib.pyplot as plt
import numpy as np
import torch
import warnings
from typing import Union, Optional
import shutil

warnings.simplefilter(action="ignore", category=FutureWarning)

sys.path.append(str(Path().resolve().parent))
from src.utils.constants import AMP_MIN, AMP_MAX, EPS
from src.utils.sar_utils import symmetrize, load_cosar
from src.utils.debug import print_statistics, print_images_statistics
from src.utils.processing_utils import clip


# Toggle if the notebook saves the raw symmetrized patcha and the MERLIN/ADAM-NOC references .npy files after creation
SAVE_NPY = True

# ----- Paths -----
CKP_PATH = Path("../logs/train/sar_ddc/")
MERLIN_CKP_PATH = Path("../data/method_ground_truths/MERLIN/checkpoints/last.ckpt")
ADAM_NOC_CKP_PATH = Path("../data/method_ground_truths/ADAM_NOC/checkpoints/last.ckpt")

# City center, around "Alsterfontäne"
HAMBURG_COS_PATH = Path(
    "../data/TSX_cos_files/Hamburg_TDX1_SAR__SSC______SM_S_SRA_20180112T165337_20180112T165345_IMAGE_HH_SRA_strip_004.cos"
)
HAMBURG_PATCH_SLICE = (slice(11000, 12024), slice(8500, 9524))
HAMBURG_PATCH_SLICE_STR = f"[{HAMBURG_PATCH_SLICE[0].start}:{HAMBURG_PATCH_SLICE[0].stop}-{HAMBURG_PATCH_SLICE[1].start}:{HAMBURG_PATCH_SLICE[1].stop}]"

if SAVE_NPY:
    STORAGE_FOLDER = Path(f"../data/visualization/Hamburg_{HAMBURG_PATCH_SLICE_STR}/")
    if STORAGE_FOLDER.exists():
        shutil.rmtree(STORAGE_FOLDER)
    STORAGE_FOLDER.mkdir(parents=True, exist_ok=True)


def show_image(
    img: np.ndarray,
    title: str,
    clip_factor: int = 3,
    dpi: int = 100,
    path_to_png: Optional[Path] = None,
) -> None:
    img_clipped = clip(img, clip_factor=clip_factor)

    fig, ax = plt.subplots(figsize=(4, 4), dpi=dpi)
    img_plot = ax.imshow(img_clipped, cmap="gray")
    fig.colorbar(img_plot, ax=ax, shrink=0.8)
    ax.set_title(title)
    ax.axis("off")
    fig.show()

    if path_to_png is not None:
        plt.imsave(path_to_png, img_clipped, cmap="gray", dpi=dpi)
        print(f"Saved image to {path_to_png}.")


def instantiate_model_from_lightning_ckpt(
    ckpt_path: Path, model_name: str = ""
) -> torch.nn.Module:
    config_path = ckpt_path.parent.parent / ".hydra" / "config.yaml"
    if not config_path.exists():
        raise FileNotFoundError(f"Training config for {model_name} not found at {config_path}.")
    print(f"Loading original {model_name} training config from {config_path}")
    cfg = OmegaConf.load(config_path)

    print(f"Instantiating model <{cfg.model._target_}>")
    model = hydra.utils.instantiate(cfg.model)
    checkpoint = torch.load(str(ckpt_path), map_location="cpu")
    msg = model.load_state_dict(checkpoint["state_dict"], strict=True)
    print(f"Loaded {model_name} checkpoint state_dict with message: {msg}")
    model.eval()

    return model


def show_and_save_image(
    img_logI: np.ndarray,
    img_linA: np.ndarray,
    save_path: Path,
    name: str,
) -> None:
    for scale in ["logI", "linA"]:
        img = img_logI if scale == "logI" else img_linA
        save_img_path = save_path / f"{scale}_{name}.npy"
        np.save(save_img_path, img)
        print(f"Saved {scale} image to {save_img_path}.")

        # Do not show or save the linA image
        if scale == "logI":
            show_image(
                img,
                f"{scale} {name} patch",
                path_to_png=save_img_path.with_suffix(".png"),
            )

In [ ]:
# --- Load and symmetrize ---
# From COS file
cos_tile = load_cosar(HAMBURG_COS_PATH, logger=None)
if cos_tile is None:
    raise ValueError(f"Failed to load {HAMBURG_COS_PATH}")
assert type(cos_tile) is np.ndarray and cos_tile.ndim == 3 and cos_tile.shape[2] == 2, (
    f"Unexpected COS tile shape: {cos_tile.shape}"
)
cos_tile = symmetrize(cos_tile)
patch_cos = cos_tile[HAMBURG_PATCH_SLICE]
print_statistics("Patch symmetrized", patch_cos)
if SAVE_NPY:
    save_sym_path = STORAGE_FOLDER / f"sym_Noisy.npy"
    np.save(save_sym_path, patch_cos)
    print(f"Saved symmetrized patch to {save_sym_path}.")

# --- Prepare patch ---
patch = torch.from_numpy(patch_cos).float()  # [H, W, 2]
real = patch[..., 0].unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
imag = patch[..., 1].unsqueeze(0).unsqueeze(0)
patch = torch.stack((real, imag), dim=1).squeeze(2)  # [1, 2, H, W]
patch_I = 0.5 * (torch.square(patch[:, 0, :, :]) + torch.square(patch[:, 1, :, :]))
patch_linA = torch.sqrt(patch_I).squeeze().detach().numpy()  # [H, W]
patch_logI = torch.log(patch_I + EPS).squeeze().detach().numpy()
print_statistics("Patch logI", patch_logI)

if SAVE_NPY:
    show_and_save_image(patch_logI, patch_linA, STORAGE_FOLDER, "Noisy")
else:
    show_image(patch_logI, "logI Noisy patch")

## MERLIN (own implementation)

In [ ]:
# Find and load the config of MERLIN
merlin_model = instantiate_model_from_lightning_ckpt(MERLIN_CKP_PATH, model_name="MERLIN")

recon_real = merlin_model.forward(real)
recon_imag = merlin_model.forward(imag)

recon_real_denorm = recon_real.squeeze() * (AMP_MAX - AMP_MIN) + AMP_MIN
recon_imag_denorm = recon_imag.squeeze() * (AMP_MAX - AMP_MIN) + AMP_MIN
recon_real_lin = torch.exp(recon_real_denorm)
recon_imag_lin = torch.exp(recon_imag_denorm)
merlin_recon = 0.5 * (torch.square(recon_real_lin) + torch.square(recon_imag_lin))
merlin_recon_logI = torch.log(merlin_recon + EPS).detach().numpy()
merlin_recon_linA = np.sqrt(merlin_recon.detach().numpy())

print_images_statistics(
    {
        "Input real": real.detach().numpy(),
        "Input imag": imag.detach().numpy(),
        "MERLIN recon real": recon_real.detach().numpy(),
        "MERLIN recon imag": recon_imag.detach().numpy(),
        "MERLIN recon real denorm": recon_real_denorm.detach().numpy(),
        "MERLIN recon imag denorm": recon_imag_denorm.detach().numpy(),
        "MERLIN recon real (Lin)": recon_real_lin.detach().numpy(),
        "MERLIN recon imag (Lin)": recon_imag_lin.detach().numpy(),
        "MERLIN recon avg": merlin_recon.detach().numpy(),
        "MERLIN recon Log-I": merlin_recon_logI,
        "MERLIN recon Lin-A": merlin_recon_linA,
    }
)

if SAVE_NPY:
    show_and_save_image(merlin_recon_logI, merlin_recon_linA, STORAGE_FOLDER, "MERLIN")
else:
    show_image(merlin_recon_logI, "Loaded MERLIN Reference")

## ADAM-NOC (ADAM No Compression)

In [ ]:
adam_model = instantiate_model_from_lightning_ckpt(ADAM_NOC_CKP_PATH, model_name="ADAM")

recon = adam_model.forward(patch)
recon = recon["x_hat"]

recon = torch.exp(recon.squeeze() * (AMP_MAX - AMP_MIN) + AMP_MIN)
adam_recon = 0.5 * (torch.square(recon[0, ...]) + torch.square(recon[1, ...]))
adam_recon_logI = torch.log(adam_recon + EPS).detach().numpy()
adam_recon_linA = np.sqrt(adam_recon.detach().numpy())

print_images_statistics(
    {
        "Input patch": patch.detach().numpy(),
        "ADAM recon": recon.detach().numpy(),
        "ADAM recon Lin": recon.detach().numpy(),
        "ADAM recon avg": adam_recon.detach().numpy(),
        "ADAM recon Log-I": adam_recon_logI,
        "ADAM recon Lin-A": adam_recon_linA,
    }
)

if SAVE_NPY:
    show_and_save_image(adam_recon_logI, adam_recon_linA, STORAGE_FOLDER, "ADAM_NOC")
else:
    show_image(adam_recon_logI, "Loaded ADAM-NOC Reference")

## MERLIN-DDS (from deepdespeckling checkpoint)

In [ ]:
# Load MERLIN-DDS
merlin_DDS_ref = np.load(
    STORAGE_FOLDER.parent / "MERLIN-DDS" / f"linA_MERLIN-DDS_Hamburg_patch_{HAMBURG_PATCH_SLICE_STR}.npy",
    allow_pickle=True,
).item()
merlin_DDS_linA = merlin_DDS_ref["denoised"]["full"]
merlin_DDS_logI = np.log(np.square(merlin_DDS_linA) + EPS)

print_images_statistics(
    {
        "MERLIN-DDS recon Lin-A": merlin_DDS_linA,
        "MERLIN-DDS recon Log-I": merlin_DDS_logI,
    }
)

if SAVE_NPY:
    show_and_save_image(merlin_DDS_logI, merlin_DDS_linA, STORAGE_FOLDER, "MERLIN-DDS")
else:
    show_image(merlin_DDS_logI, "Loaded MERLIN-DDS Reference")

## Compare all reconstructions

In [ ]:
# Compare all refs together: subplot 2x2
fig, axs = plt.subplots(2, 2, figsize=(10, 10), dpi=300)
axs[0, 0].imshow(clip(patch_logI, clip_factor=3), cmap="gray")
axs[0, 0].set_title("Noisy")
axs[0, 0].axis("off")

axs[0, 1].imshow(clip(merlin_recon_logI, clip_factor=3), cmap="gray")
axs[0, 1].set_title("MERLIN")
axs[0, 1].axis("off")

axs[1, 0].imshow(clip(adam_recon_logI, clip_factor=3), cmap="gray")
axs[1, 0].set_title("ADAM-NOC")
axs[1, 0].axis("off")

axs[1, 1].imshow(clip(merlin_DDS_logI, clip_factor=3), cmap="gray")
axs[1, 1].set_title("MERLIN Deep Despeckling")
axs[1, 1].axis("off")

fig.suptitle("Comparison of Denoising References", fontsize=16)
fig.tight_layout()
fig.show()

comparison_path = STORAGE_FOLDER / "comparison_denoised_references.png"
plt.imsave(comparison_path, fig.canvas.buffer_rgba())
print(f"Saved comparison figure to {comparison_path}.")